In [1]:
from dotenv import load_dotenv
import os
import base64
from requests import post, get
import json
import pandas as pd  # 👈 pour le DataFrame
from urllib.parse import quote

pd.set_option('display.max_rows', None)

load_dotenv()

client_id = "1ef0771d66de447f9aa9d5de33ac1968"
client_secret = "23efdfeca3b04f908a9e5ad427c3748a"


def get_token():
    auth_string = client_id + ":" + client_secret
    auth_bytes = auth_string.encode("utf-8")
    auth_base64 = base64.b64encode(auth_bytes).decode("utf-8")

    url = "https://accounts.spotify.com/api/token"
    headers = {
        "Authorization": "Basic " + auth_base64,
        "Content-Type": "application/x-www-form-urlencoded"
    }
    data = {"grant_type": "client_credentials"}

    # NE PAS mettre verify=False ici
    result = post(url, headers=headers, data=data, verify=False)
    print("Status token:", result.status_code)
    print("Réponse brute:", result.text)   # pour debug

    result.raise_for_status()
    json_result = result.json()
    token = json_result["access_token"]
    return token

def get_auth_header(token):
    return {"Authorization": "Bearer " + token}

# def search_for_artist(token, artist_name):
#     url = "https://api.spotify.com/v1/search"
#     headers = get_auth_header(token)
#     query = f"?q={artist_name}&type=artist&limit=1"

#     query_url = url + query
#     result = get(query_url, headers=headers, verify=False)
#     print("Status search:", result.status_code)
#     json_result = json.loads(result.content)["artists"]["items"]
#     if len(json_result) == 0:
#         print("No artist found")
#         return None

#     return json_result[0]

# def get_songs_by_artist(token, artist_id):
#     url = f"https://api.spotify.com/v1/artists/{artist_id}/top-tracks?country=FR"
#     headers = get_auth_header(token)
#     result = get(url, headers=headers, verify=False)
#     print("Status songs:", result.status_code)
#     json_result = json.loads(result.content)["tracks"]
#     return json_result

def search_for_playlist(token, playlist_name):
    """Recherche une playlist par nom et renvoie le premier résultat."""
    url = "https://api.spotify.com/v1/search"
    headers = get_auth_header(token)
    # encoder le nom pour gérer espaces/accents
    query = f"?q={playlist_name}&type=playlist&market=FR&limit=5"
    query_url = url + query
    result = get(query_url, headers=headers, verify=False)
    print("Status search playlist:", result.status_code)
    json_result = json.loads(result.content)
    if len(json_result) == 0:
        print("No playlist found")
        return None
    return json_result

def get_tracks_from_playlist(token, playlist_id):
    """Récupère les pistes d'une playlist (jusqu'à 100, suffisant pour le Top 50)."""
    url = f"https://api.spotify.com/v1/playlists/{playlist_id}/tracks?limit=100"
    headers = get_auth_header(token)
    result = get(url, headers=headers, verify=False)
    print("Status playlist tracks:", result.status_code)
    json_result = json.loads(result.content)
    items = json_result.get("items", [])
    # on renvoie la liste d'objets 'track' (comme pour l'artiste)
    tracks = [it["track"] for it in items if it.get("track") is not None]
    return tracks

token = get_token()
# result = search_for_artist(token, "SCH")
# artist_id = result["id"]
# songs = get_songs_by_artist(token, artist_id)

playlist = search_for_playlist(token, "Top%2050%20:%20France")
print(playlist)
# playlist_id = playlist["id"]
tracks = get_tracks_from_playlist(token, "2IgPkhcHbgQ4s4PdCxljAx")

df=pd.DataFrame()

id_list=[]
track_list=[]
artist_list=[]
feat_list=[]
cover_list=[]
popularity_list=[]
relase_date_list=[]
feat_list_temp=[]


for idx, track in enumerate(tracks):
  id_list.append(track['id'])
  track_list.append(track['name'])
  artist_list.append(track['artists'][0]['name'])
  if len(track['artists'])>1 :
    for idx2, artist in enumerate(track['artists']) :
      if idx2>0 :
        feat_list_temp.append(artist['name'])
    feat_list.append(', '.join(feat_list_temp))
    feat_list_temp.clear()
  else :
    feat_list.append('None')
  cover_list.append(track['album']['images'][0]['url'])
  relase_date_list.append(track['album']['release_date'])
  popularity_list.append(track['popularity'])

df['id']=id_list
df['track']=track_list
df['artist']=artist_list
df['feat']=feat_list
df['cover']=cover_list
df['release_date']=relase_date_list
df['popularity']=popularity_list
display(df)




C:\Users\ndesbrosse\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'accounts.spotify.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status token: 200
Réponse brute: {"access_token":"BQB4gs97E5zLDIsIpIh9owxUhEgCU_XGOO35IBz3dABypX0-2MevemwhddGshkPK3TfAUu5v5UjLqQD-uzIDWxfdbfMVTJXvZbdNAZJ6TOlRlPzbcyNUxIRXcEXilwEGuz6jQ5TlkYI","token_type":"Bearer","expires_in":3600}


C:\Users\ndesbrosse\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.spotify.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status search playlist: 200
{'playlists': {'href': 'https://api.spotify.com/v1/search?offset=0&limit=5&query=Top%2050%20%3A%20France&type=playlist&market=FR', 'limit': 5, 'next': 'https://api.spotify.com/v1/search?offset=5&limit=5&query=Top%2050%20%3A%20France&type=playlist&market=FR', 'offset': 0, 'previous': None, 'total': 899, 'items': [None, None, None, None, None]}}


C:\Users\ndesbrosse\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.spotify.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status playlist tracks: 200


,id,track,artist,feat,cover,release_date,popularity
0,5Y8C6KjzBRKvcT3Aln1Bc4,melodrama,disiz,Theodora,https://i.scdn.co/image/ab67616d0000b273ee5f01...,2025-09-26,82
1,5gvvdf0mk5nL5Gu9h5cUTX,PARISIENNE,GIMS,La Mano 1.9,https://i.scdn.co/image/ab67616d0000b273d6455d...,2025-02-20,67
2,2lwLLpCLIQ6lc5EvvdjG5C,Ailleurs,Orelsan,None,https://i.scdn.co/image/ab67616d0000b2731231ef...,2025-11-07,64
3,0DAhk47iSMkfPKm5MDW17x,Un monde à l'autre,GP Explorer,"GIMS, La Mano 1.9, SCH",https://i.scdn.co/image/ab67616d0000b273387935...,2025-09-11,77
4,31TXxq8gfgYyrYClnYY48m,The Fate of Ophelia,Taylor Swift,None,https://i.scdn.co/image/ab67616d0000b273b034f9...,2025-10-03,88
5,16nyxKShgXn5qrh9zaHCGX,Adriano,Niska,None,https://i.scdn.co/image/ab67616d0000b2737f6063...,2025-07-11,77
6,1yGJ40k7HLF3odITZPoQs4,MON BÉBÉ,RnBoi,None,https://i.scdn.co/image/ab67616d0000b273deb707...,2025-10-24,77
7,32cwemsMJdzzsOYjVzfYMS,Mauvais Garçon,Helena,None,https://i.scdn.co/image/ab67616d0000b273108597...,2024-11-20,60
8,5DTahoMKuix8Fi1WSXGwIW,VIANO,RK,Genezio,https://i.scdn.co/image/ab67616d0000b27320038f...,2025-07-18,78
9,2T8yuUKl1nhmtaIocqWo4i,Berghain,ROSALÍA,"Björk, Yves Tumor",https://i.scdn.co/image/ab67616d0000b27393ee2e...,2025-11-07,75


In [24]:
def get_audio_features_from_track(token, track_id):
    """Récupère les pistes d'une playlist (jusqu'à 100, suffisant pour le Top 50)."""
    url = f"https://api.spotify.com/v1/tracks/{track_id}"
    headers = get_auth_header(token)
    result = get(url, headers=headers, verify=False)
    print("Status playlist tracks:", result.status_code)
    json_result = json.loads(result.content)
    # on renvoie la liste d'objets 'track' (comme pour l'artiste)
    return json_result

# audio_features_list = []

# for track_id in df['id']:
#     features = get_audio_features_from_track(token, track_id)
#     audio_features_list.append(features)

# display(audio_features_list)

print(get_audio_features_from_track(token, "1vdXYpwDTZFgVc1inmirQ7"))


Status playlist tracks: 200
{'album': {'album_type': 'album', 'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/0GOx72r5AAEKRGQFn3xqXK'}, 'href': 'https://api.spotify.com/v1/artists/0GOx72r5AAEKRGQFn3xqXK', 'id': '0GOx72r5AAEKRGQFn3xqXK', 'name': 'GIMS', 'type': 'artist', 'uri': 'spotify:artist:0GOx72r5AAEKRGQFn3xqXK'}], 'available_markets': [], 'external_urls': {'spotify': 'https://open.spotify.com/album/1eyeaGRt8dRt5wm7Wg7Fyc'}, 'href': 'https://api.spotify.com/v1/albums/1eyeaGRt8dRt5wm7Wg7Fyc', 'id': '1eyeaGRt8dRt5wm7Wg7Fyc', 'images': [{'url': 'https://i.scdn.co/image/ab67616d0000b2739568fc14ab5f80eb1214388b', 'width': 640, 'height': 640}, {'url': 'https://i.scdn.co/image/ab67616d00001e029568fc14ab5f80eb1214388b', 'width': 300, 'height': 300}, {'url': 'https://i.scdn.co/image/ab67616d000048519568fc14ab5f80eb1214388b', 'width': 64, 'height': 64}], 'name': "LE NORD SE SOUVIENT : L'ODYSSÉE", 'release_date': '2025-06-25', 'release_date_precision': 'day', 'total

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.spotify.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
